In [24]:
import polars as pl

import io
import chess.pgn
import re
from typing import Dict, Optional

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [25]:
INPUT = '../../../data/sample_data.pgn'

In [26]:
def parse_pgn_to_dataframe(pgn_text):
    """
    Parse a PGN (Portable Game Notation) file and convert it to a Polars DataFrame.
    
    Args:
        pgn_text (str): The contents of a PGN file as a string
        
    Returns:
        pl.DataFrame: A DataFrame containing parsed game information
    """
    games = []
    
    # Split the text into individual games
    game_blocks = re.split(r'\n\n(?=\[Event)', pgn_text.strip())
    
    for block in game_blocks:
        if not block.strip():
            continue
            
        game_data = {}
        
        # Extract header information
        headers = re.findall(r'\[(\w+)\s+"([^"]*)"\]', block)
        for key, value in headers:
            game_data[key] = value
        
        # Extract moves (everything after the last header until result)
        moves_match = re.search(r'\]\n\n(.+?)(?:\s+(?:1-0|0-1|1/2-1/2|\*))?$', block, re.DOTALL)
        if moves_match:
            moves_text = moves_match.group(1).strip()
            # Remove clock annotations and extra whitespace
            moves_clean = re.sub(r'\{[^}]*\}', '', moves_text)
            moves_clean = re.sub(r'\s+', ' ', moves_clean).strip()
            game_data['Moves'] = moves_clean
        else:
            game_data['Moves'] = ''
        
        games.append(game_data)
    
    # Create DataFrame
    if games:
        df = pl.DataFrame(games)
        
        # Convert numeric columns to appropriate types
        numeric_columns = ['WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff']
        for col in numeric_columns:
            if col in df.columns:
                df = df.with_columns(
                    pl.col(col).cast(pl.Int32, strict=False)
                )

        formats_to_keep = ["Rated Blitz game",
                       "Rated Bullet game",
                       "Rated Rapid game"]

        df = df.filter(pl.col('Event').str.contains('|'.join(formats_to_keep)))

        df = df.filter(pl.col('Termination') != 'Abandoned')

        df = df.with_columns(
        pl.when(pl.col("Event").str.contains("Rapid")).then(pl.lit("Rapid"))
        .when(pl.col("Event").str.contains("Bullet")).then(pl.lit("Bullet"))
        .when(pl.col("Event").str.contains("Blitz")).then(pl.lit("Blitz"))
        .otherwise(pl.col("Event"))
        .alias("TimeControl")
        )

        df = df.with_columns(
            pl.col("Result").replace({"1-0": 1, "0-1": -1, "1/2-1/2": 0}).alias("Result")
        ).cast({"Result": pl.Int8})
    

        # calculate the number of moves
        def calculate_num_moves(moves: str) -> int:
            game = chess.pgn.read_game(io.StringIO(moves))
            if game is None:
                return 0
            num_moves = sum(1 for _ in game.mainline_moves()) // 2
            return num_moves
        
        df = df.with_columns(
            pl.col('Moves').map_elements(calculate_num_moves, return_dtype=pl.Int32).alias('NumMoves')
        )


        df = df.select(['TimeControl','WhiteElo', 'BlackElo',
                    'Moves','Result', 'NumMoves'])


        return df
    else:
        return pl.DataFrame()

# Read PGN file
with open(INPUT, 'r', encoding='utf-8') as file:
    pgn_text = file.read()
    df = parse_pgn_to_dataframe(pgn_text)

# 35, 27, 31

df.head()

TimeControl,WhiteElo,BlackElo,Moves,Result,NumMoves
str,i32,i32,str,i8,i32
"""Bullet""",1706,1671,"""1. d4 1... c5 2. e3 2... e6 3.…",-1,35
"""Bullet""",2262,2191,"""1. d4 1... d6 2. Nf3 2... Nf6 …",1,26
"""Bullet""",2279,2339,"""1. d4 1... Nf6 2. Nf3 2... e6 …",-1,31
"""Bullet""",971,1040,"""1. b4 1... e5 2. Bb2 2... f5 3…",-1,25
"""Bullet""",1752,1737,"""1. e4 1... e5 2. Nf3 2... d6 3…",-1,41


In [27]:
class FeatureEngineer:
    """Extract chess features from PGN moves at a specific move number."""
    
    PIECE_VALUES = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0
    }
    
    @staticmethod
    def get_board_at_move(moves: str, target_move: int = 15) -> Optional[chess.Board]:
        """
        Get the board state at a specific move number.
        
        Args:
            moves: PGN moves string
            target_move: Move number to analyze (default: 15)
            
        Returns:
            chess.Board object at the target move, or None if move doesn't exist
        """
        game: Optional[chess.pgn.Game] = chess.pgn.read_game(io.StringIO(moves))
        if game is None:
            return None


        board = game.board()
        move_count = 0
        
        for move in game.mainline_moves():
            board.push(move)
            move_count += 1
            # Each pair of moves (white + black) = 1 full move
            if move_count == target_move * 2:
                return board
        return board

    @staticmethod
    def calculate_material(board: chess.Board) -> Dict[str, int]:
        """
        Calculate material count for both sides.
        
        Args:
            board: Chess board state
            
        Returns:
            Dictionary with white_material, black_material, and material_diff
        """
        white_material: int = 0
        black_material: int = 0
        
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                value: int = FeatureEngineer.PIECE_VALUES[piece.piece_type]
                if piece.color == chess.WHITE:
                    white_material += value
                else:
                    black_material += value
        
        return {
            'white_material': white_material,
            'black_material': black_material,
            'material_diff': white_material - black_material
        }
    @staticmethod
    def extract_all_features(moves: str, target_move: int = 15) -> Dict[str, Optional[int]]:
        """
        Extract all chess features for a game at a specific move.
        
        Args:
            moves: PGN moves string
            target_move: Move number to analyze (default: 15)
            
        Returns:
            Dictionary with all features, or None values if move doesn't exist
        """
        board = FeatureEngineer.get_board_at_move(moves, target_move)
        
        if board is None:
            # Return None for all features if board state unavailable
            return {
                'white_material': None,
                'black_material': None,
                'material_diff': None,

            }
        
        features = {}
        features.update(FeatureEngineer.calculate_material(board))


        return features
    

In [28]:
def add_chess_features(df: pl.DataFrame, target_move: int = 15):
    """
    Add chess features to the dataframe.
    
    Args:
        df: Polars DataFrame with 'Moves' column
        target_move: Move number to analyze (default: 15)
        
    Returns:
        DataFrame with added feature columns
    """
    
    # Extract features for all games
    features = df.select("Moves").with_columns(
        pl.col("Moves")
        .map_elements(
            lambda moves: FeatureEngineer.extract_all_features(moves, target_move),
            return_dtype=pl.Struct({
                'white_material': pl.Int64,
                'black_material': pl.Int64,
                'material_diff': pl.Int64,
            })
        )
        .alias("features")
    )
    
    # Unnest the struct to separate columns
    features = features.unnest("features")
    
    # Concatenate with original dataframe
    df_with_features = pl.concat([df, features.drop("Moves")], how="horizontal")
    
    return df_with_features

In [29]:
df_fe = add_chess_features(df, target_move=15)
df_fe.head()

TimeControl,WhiteElo,BlackElo,Moves,Result,NumMoves,white_material,black_material,material_diff
str,i32,i32,str,i8,i32,i64,i64,i64
"""Bullet""",1706,1671,"""1. d4 1... c5 2. e3 2... e6 3.…",-1,35,34,35,-1
"""Bullet""",2262,2191,"""1. d4 1... d6 2. Nf3 2... Nf6 …",1,26,32,31,1
"""Bullet""",2279,2339,"""1. d4 1... Nf6 2. Nf3 2... e6 …",-1,31,37,38,-1
"""Bullet""",971,1040,"""1. b4 1... e5 2. Bb2 2... f5 3…",-1,25,26,23,3
"""Bullet""",1752,1737,"""1. e4 1... e5 2. Nf3 2... d6 3…",-1,41,32,31,1


In [30]:
df_final = df_fe.drop(['Moves'])
df_final

TimeControl,WhiteElo,BlackElo,Result,NumMoves,white_material,black_material,material_diff
str,i32,i32,i8,i32,i64,i64,i64
"""Bullet""",1706,1671,-1,35,34,35,-1
"""Bullet""",2262,2191,1,26,32,31,1
"""Bullet""",2279,2339,-1,31,37,38,-1
"""Bullet""",971,1040,-1,25,26,23,3
"""Bullet""",1752,1737,-1,41,32,31,1
…,…,…,…,…,…,…,…
"""Blitz""",1513,1598,-1,28,21,19,2
"""Blitz""",1370,1377,1,34,37,37,0
"""Blitz""",2177,2169,1,35,39,39,0


In [31]:
X = df_final.drop('Result')
y = df_final['Result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [32]:
cat_cols = ['TimeControl']
num_cols = X_train.drop(['TimeControl']).columns

In [33]:
pipeline = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

full_pipeline = Pipeline([
    ('preprocessor', pipeline),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
classes = y.unique().to_list()
print(classification_report(y_test, y_pred, target_names=[str(c) for c in classes]))

              precision    recall  f1-score   support

          -1       0.55      0.45      0.50        66
           0       0.00      0.00      0.00         5
           1       0.56      0.68      0.61        72

    accuracy                           0.55       143
   macro avg       0.37      0.38      0.37       143
weighted avg       0.53      0.55      0.54       143



In [34]:
confusion_matrix(y_test, y_pred, labels=[1, 0, -1])

array([[49,  0, 23],
       [ 3,  0,  2],
       [36,  0, 30]])

In [35]:
# Get feature names from the pipeline
cat_feature_names = full_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
num_feature_names = num_cols
feature_names = list(cat_feature_names) + list(num_feature_names)

# Get feature importances from the trained RandomForest
importances = full_pipeline.named_steps['classifier'].feature_importances_

# Combine feature names and importances
feature_importance = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)

# Display feature importances
for name, importance in feature_importance:
    print(f"{name}: {importance:.4f}")

BlackElo: 0.2071
NumMoves: 0.2037
WhiteElo: 0.2013
black_material: 0.1105
white_material: 0.1092
material_diff: 0.1060
TimeControl_Bullet: 0.0241
TimeControl_Blitz: 0.0213
TimeControl_Rapid: 0.0169
